In [3]:
!pip install dash
!pip install comm
from dash import Dash, html, dcc, Input, Output
import comm
import plotly.express as px

# Load data
df = px.data.gapminder()

app = Dash(__name__)

#human-readable numbers
def format_pop(n):
    if n >= 1e9:
        return f"{n/1e9:.2f} B"
    elif n >= 1e6:
        return f"{n/1e6:.2f} M"
    elif n >= 1e3:
        return f"{n/1e3:.1f} K"
    return str(n)

# Ordinal helper
def ordinal(n):
    if 10 <= n % 100 <= 20:
          suffix = "th"
    else:
          suffix = {1: "st", 2: "nd", 3: "rd"}.get(n % 10, "th")
    return f"{n}{suffix}"

app.layout = html.Div([

    # TOP ROW
    html.Div([
        html.Div([
            html.H2("Development of countries in")
        ], style={"width": "50%", "display": "inline-block"}),

        html.Div([
            dcc.Dropdown(
                id="continent_dropdown",
                options=[{"label": c, "value": c} for c in df["continent"].unique()],
                value="Europe",
                clearable=False
            )
        ], style={"width": "30%", "display": "inline-block"})
    ]),

    # BOTTOM ROW
    html.Div([

        # LEFT: animated bubble plot
        html.Div([
            dcc.Graph(id="bubble_plot")
        ], style={"width": "60%", "display": "inline-block"}),

        # RIGHT: table
        html.Div([
            html.H4("Top 5 countries by population"),
            html.Div(id="table_output") ,
        dcc.Slider(
          id="year_slider",
          min=df["year"].min(),
          max=df["year"].max(),
          step=5,
          value=1952,
          marks={str(year): str(year) for year in df["year"].unique()}
        )
        ], style={"width": "35%", "display": "inline-block", "verticalAlign": "top"})

    ]), # <--- Added comma here
    # SECOND ROW
    html.Div([

    # LEFT: text
      html.Div([
          html.H4("Let's focus on")
      ], style={"width": "50%", "display": "inline-block"}),

    # RIGHT: country dropdown
      html.Div([
        dcc.Dropdown(
            id="country_dropdown",
            options=[],   # will be filled dynamically
            value=None,
            clearable=False
        )
      ], style={"width": "30%", "display": "inline-block"})

  ]),
    html.Div([

    # LEFT: choropleth map
    html.Div([
        dcc.Graph(id="map_plot")
    ], style={"width": "48%", "display": "inline-block"}),

    # RIGHT: text
    html.Div([
        html.Div(id="text_panel")
    ], style={"width": "48%", "display": "inline-block", "verticalAlign": "top"})

]),
  html.Div([
    dcc.Graph(id="comparison_plot")
], style={"width": "100%", "marginTop": "30px"})

])


# Bubble plot (filtered by continent)
@app.callback(
    Output("bubble_plot", "figure"),
    Input("continent_dropdown", "value")
)

def update_plot(continent):

    dff = df[df["continent"] == continent]

    fig = px.scatter(
        dff,
        x="gdpPercap",
        y="lifeExp",
        size="pop",
        color="country",
        hover_name="country",
        animation_frame="year",
        log_x=True,
        range_y=[20,100],
        range_x=[100,100000],
        size_max=60,
        title=f"{continent}: Life expectancy vs GDP per capita"
    )

    return fig


# Table with largest countries by year
@app.callback(
    Output("table_output", "children"),
    Input("continent_dropdown", "value"),
    Input("year_slider", "value")
)
def update_table(continent, year):

    dff = df[(df["continent"] == continent) & (df["year"] == year)]
    top5 = dff.nlargest(5, "pop")[["country", "pop"]]

    rows = [
        html.Tr([
            html.Td(row["country"]),
            html.Td(format_pop(row["pop"]))
        ])
        for _, row in top5.iterrows()
    ]

    return html.Div([
        html.P(f"Year: {year}"),
        html.Table([
            html.Thead(html.Tr([html.Th("Country"), html.Th("Population")])),
            html.Tbody(rows)
        ])
    ])
@app.callback(
    Output("country_dropdown", "options"),
    Output("country_dropdown", "value"),
    Input("continent_dropdown", "value")
)
def update_country_dropdown(continent):

    dff = df[df["continent"] == continent]
    countries = sorted(dff["country"].unique())

    options = ([{"label": c, "value": c} for c in countries])

    # default: first country
    value = countries[0] if countries else None

    return options, value
@app.callback(
    Output("map_plot", "figure"),
    Input("continent_dropdown", "value"),
    Input("country_dropdown", "value"),
    Input("year_slider", "value")
)
def update_map(continent, country, year):

    dff = df[(df["continent"] == continent) & (df["year"] == year)].copy()

    # Create highlight column
    dff["highlight"] = dff["country"].apply(
        lambda x: 1 if x == country else 0
    )

    fig = px.choropleth(
        dff,
        locations="iso_alpha",
        color="highlight",
        hover_name="country",
        color_continuous_scale=["lightgray", "red"],
        range_color=[0, 1]
      # title=f"{country}"
    )

    # Clean up color bar
    fig.update_coloraxes(showscale=False)

    fig.update_layout(height=500)

    return fig
@app.callback(
    Output("text_panel", "children"),
    Input("country_dropdown", "value"),
    Input("continent_dropdown", "value"),
    Input("year_slider", "value")
)
def update_text_panel(country, continent, year):

    # --- Continent ranking ---
    dff_cont = df[(df["continent"] == continent) & (df["year"] == year)]
    dff_cont = dff_cont.sort_values("pop", ascending=False).reset_index(drop=True)
    rank_cont = dff_cont.index[dff_cont["country"] == country][0] + 1

    # --- World ranking ---
    dff_world = df[df["year"] == year]
    dff_world = dff_world.sort_values("pop", ascending=False).reset_index(drop=True)
    rank_world = dff_world.index[dff_world["country"] == country][0] + 1

    # --- Country-specific values ---
    dff_country = df[(df["country"] == country) & (df["year"] == year)]
    gdp = dff_country["gdpPercap"].values[0]
    life_exp = dff_country["lifeExp"].values[0]

    return html.Div([
        html.H3(f"In {year}"),
        html.P(f"{country} is the {ordinal(rank_cont)} most populous country in {continent}."),
        html.P(f"It is also the {ordinal(rank_world)} most populous country in the world."),
        html.P(f"Its GDP per capita is ${gdp:,.0f}."),
        html.P(f"And the life expectancy of a person living there is {life_exp:.1f} years.")
    ])
@app.callback(
    Output("comparison_plot", "figure"),
    Input("continent_dropdown", "value"),
    Input("country_dropdown", "value")
)
def update_comparison_plot(continent, country):
#Find "nearest neighbors" in terms of population
    # --- Step 1: data for 1952 ---
    base = df[(df["continent"] == continent) & (df["year"] == 1952)]

    # population of selected country in 1952
    pop_selected = base[base["country"] == country]["pop"].values[0]

    # compute similarity
    base["diff"] = abs(base["pop"] - pop_selected)

    # exclude selected country, pick 5 closest
    similar = (
        base[base["country"] != country]
        .nsmallest(10, "diff")["country"]
        .tolist()
    )

    countries_to_plot = similar + [country]

    # --- Step 2: full time data ---
    dff = df[df["country"].isin(countries_to_plot)]

    # --- Step 3: plot ---
    fig = px.scatter(
        dff,
        x="gdpPercap",
        y="lifeExp",
        size="pop",
        color="country",
        animation_frame="year",
        log_x=True,
        range_y=[20,100],
        range_x=[100,100000],
        size_max=60,
        title=f"Development of {country} vs similarly populated countries in {continent}"
    )

    # highlight selected country
    for trace in fig.data:
        if trace.name == country:
            trace.marker.line.width = 3
        else:
            trace.marker.opacity = 0.5

    fig.update_layout(height=600)

    return fig

if __name__ == "__main__":
    app.run(port=8062, debug=True, jupyter_mode='external')
    #output.serve_kernel_port_as_iframe(8062)

Dash app running on http://127.0.0.1:8062/


/var/folders/8p/ym97qvfj5p3cqhq47g7vmvgm0000gn/T/ipykernel_39875/3218653780.py:255: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/8p/ym97qvfj5p3cqhq47g7vmvgm0000gn/T/ipykernel_39875/3218653780.py:255: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/8p/ym97qvfj5p3cqhq47g7vmvgm0000gn/T/ipykernel_39875/3218653780.py:255: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in